# RSNA Knee Abnormality Detection: Exploratory Data Analysis & Clinical Forensics
---

## Calisma Ozeti ve Metodoloji

Bu notebook, Kaggle **RSNA Knee Abnormality Detection** veri kumesinin anatomik, metinsel, sekans ve etiket yapisini arastirma duzeyinde analiz etmek uzere hazirlanmistir.

Her analiz adimi uc temel soruya yanit veren pedagojik ve ogretici bir cerceveyle sunulmaktadir:
1. **Metod ve Arac:** Hangi analitik yontem, kutuphane ve matematiksel formul kullanildi?
2. **Klinik ve Matematiksel Mantik:** Bu inceleme neden yapildi? Tiptaki ve makine ogrenmesindeki karsiligi nedir?
3. **Bulgular ve Modelleme Karari:** Elde edilen ciktilar derin ogrenme mimarimizi ve veri on isleme boru hattimizi nasil sekillendirecek?

### Analiz Akisi:
- **Bolum 1:** Veri Kumesi Mimarisi ve Duzlem Tamligi (Plane Completeness)
- **Bolum 2:** Etiket Dagilimi ve Klinik Carpraz Risk Analizi (Disease Synergy)
- **Bolum 3:** Cok Dilli Klinik Metin Adli Bilimi (Multilingual NLP Forensics)
- **Bolum 4:** Manyetik Rezonans Fizigi: Sivi ve Yag Baskilama Sekanslari
- **Bolum 5:** 2.5D Pencereleme (Windowing) Geometrisi ve Dilim Havuzlama Mantigi
- **Bolum 6:** Modelleme Blueprint ve Stratejik Kararlar Tablosu

In [ ]:
import os
import re
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Profesyonel Grafik Yapilandirmasi
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('default')
plt.rcParams.update({
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 14,
    'figure.dpi': 100
})

# Dizin Tanimlari
NOTEBOOK_DIR = Path.cwd()
DATA_DIR = Path('../data') if (Path('../data').exists()) else Path('data')

train_df = pd.read_csv(DATA_DIR / 'train.csv')
train_series = pd.read_csv(DATA_DIR / 'train_series.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
test_series = pd.read_csv(DATA_DIR / 'test_series.csv')
sample_sub = pd.read_csv(DATA_DIR / 'sample_submission.csv')

TARGET_COLS = [c for c in sample_sub.columns if c != 'StudyInstanceUID']
print(f"Veri kumesi basariyla yuklendi. Toplam hedef sinif sayisi: {len(TARGET_COLS)}")

## Bolum 1: Veri Kumesi Mimarisi ve Duzlem Tamligi

### Metod ve Arac
`train_series.csv` uzerinde `pd.pivot_table` ve `crosstab` kullanilarak her bir hasta incelemesinde (`StudyInstanceUID`) mevcut olan seriler `Anatomical_Plane` (Sagittal, Coronal, Axial) bazinda frekans matrisine donusturuldu.

### Klinik ve Matematiksel Mantik
Cok serili MRG (Magnetic Resonance Imaging) problemlerinde en buyuk mimari zorluk **Eksik Modalite (Missing Modality)** durumudur. Eger bir hastada belirli bir duzlem eksikse (ornegin sadece Sagittal varken Axial yoksa), o duzleme bagli patolojiler (ornegin Patellofemoral kireclenme) icin ozel tamamlama (imputation) algoritmalarina veya duzlem-bagimsiz mimarilere ihtiyac duyulur. Bu yuzden veri kumesinde duzlem tamlik oranini matematiksel olarak ispatlamaliyiz.

In [ ]:
# Duzlem Tamlik Matrisi
plane_pivot = train_series.pivot_table(
    index='StudyInstanceUID',
    columns='Anatomical_Plane',
    values='SeriesInstanceUID',
    aggfunc='count',
    fill_value=0
)

completeness = ((plane_pivot['Sagittal'] > 0) & (plane_pivot['Coronal'] > 0) & (plane_pivot['Axial'] > 0)).mean() * 100
print(f"Sagittal + Coronal + Axial Duzlemlerinin Ucune de Eksiksiz Sahip Vaka Orani: %{completeness:.2f}")

# Hasta basina seri sayisi istatistikleri
series_per_study = train_series.groupby('StudyInstanceUID')['SeriesInstanceUID'].count()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Duzlem bazinda ortalama seri sayilari
plane_means = (plane_pivot > 0).mean() * 100
axes[0].bar(plane_means.index, plane_means.values, color=['#2b5c8f', '#4682b4', '#6baed6'], edgecolor='black', alpha=0.85)
axes[0].set_title('Duzlem Bazinda Vaka Kapsami (%)', weight='bold')
axes[0].set_ylabel('Kapsam Yuzdesi (%)')
axes[0].set_ylim(0, 115)
for i, v in enumerate(plane_means.values):
    axes[0].text(i, v + 2, f"%{v:.1f}", ha='center', weight='bold')

# Seri sayisi histogrami
axes[1].hist(series_per_study, bins=range(3, 16), color='#2e8b57', edgecolor='black', alpha=0.8, align='left')
axes[1].set_title('Hasta Basina Dusen Toplam Seri Sayisi Dagilimi', weight='bold')
axes[1].set_xlabel('Seri Sayisi')
axes[1].set_ylabel('Hasta Sayisi')
axes[1].axvline(series_per_study.mean(), color='red', linestyle='--', label=f'Ortalama: {series_per_study.mean():.2f}')
axes[1].legend()

plt.tight_layout()
plt.show()

### Bulgular ve Modelleme Karari
- **Veri Birligi:** Hastalarin **%100.00'i istisnasiz olarak Sagittal, Coronal ve Axial duzlemlerin ucune de sahiptir.**
- **Hacim Varyasyonu:** Hasta basina ortalama 5.53 seri dusmekte olup, en az 3, en fazla 14 seri bulunmaktadir.
- **Karar:** Eksik modalite yonetimi ile vakit kaybedilmeyecektir. Model mimarimiz, her calisma icin 3 ana duzlemi es zamanli girdi olarak kabul eden **3-Kollu Multi-View Fusion** mimarisi olarak tasarlanacaktir.

## Bolum 2: Etiket Dagilimi ve Klinik Carpraz Risk Analizi

### Metod ve Arac
Veri setindeki 58 manuel dogrulanmis altin vaka filtrelenerek pozitiflik oranlari cikarilmis, etiketler arasi Pearson korelasyon matrisi olusturulmus ve kosullu olasilik analizi yapilmistir:

$$\text{Risk Ratio} = \frac{P(\text{Hastalik}_j \mid \text{ACL}=1)}{P(\text{Hastalik}_j \mid \text{ACL}=0)}$$

### Klinik ve Matematiksel Mantik
Diz biyomekanigi birbirine bagli kokenlere dayanir. Akut on capraz bag (ACL) travmasi olustugunda, tibia kemigi one dogru sublukse olur ve lateral femoral kondil ile carpisir. Bu durum kemik ezilmesine (contusion) ve meniskus yirtiklarina yol acar. Bu sinerjiyi tespit etmek, modelin cok-gorevli ogrenme (multi-task learning) kapasitesini olcmemizi saglar.

In [ ]:
gold_mask = train_df['ACL'].notna()
gold_df = train_df[gold_mask].copy()
unlabeled_df = train_df[~gold_mask].copy()

print(f"Gold-Standard Etiketli Calisma : {len(gold_df)} (%{len(gold_df)/len(train_df)*100:.2f})")
print(f"Etiketsiz Rapor Calismasi       : {len(unlabeled_df)} (%{len(unlabeled_df)/len(train_df)*100:.2f})")

# 12 Sinifin Pozitiflik Dagilimi
target_means = gold_df[TARGET_COLS].mean().sort_values(ascending=False)

plt.figure(figsize=(11, 5))
bars = plt.barh(target_means.index, target_means.values * 100, color='#1f77b4', edgecolor='black', alpha=0.85)
plt.title('58 Altin Vakada Hastalik Pozitiflik Prevalansi (%)', weight='bold')
plt.xlabel('Pozitiflik Orani (%)')
for i, bar in enumerate(bars):
    val = target_means.values[i] * 100
    count = int(round(val * len(gold_df) / 100))
    plt.text(val + 1, bar.get_y() + bar.get_height()/2, f"%{val:.1f} ({count}/{len(gold_df)})")
plt.xlim(0, 75)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### Hastaliklar Arasi Korelasyon ve Kosullu Risk Analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 1. Korelasyon Isı Haritasi
corr = gold_df[TARGET_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='vlag', vmin=-0.6, vmax=0.6, ax=axes[0], cbar_kws={'label': 'Pearson r'})
axes[0].set_title('12 Patoloji Arasi Korelasyon Matrisi', weight='bold')

# 2. ACL Kosullu Risk Carpani
acl_pos = gold_df[gold_df['ACL'] == 1.0]
acl_neg = gold_df[gold_df['ACL'] == 0.0]
synergy_records = []

for t in TARGET_COLS:
    if t != 'ACL':
        p1 = acl_pos[t].mean()
        p0 = acl_neg[t].mean()
        ratio = p1 / (p0 + 1e-5)
        synergy_records.append({'Hastalik': t, 'ACL_Pozitif_Risk': p1 * 100, 'ACL_Negatif_Risk': p0 * 100, 'Risk_Carpani': ratio})

synergy_df = pd.DataFrame(synergy_records).sort_values(by='Risk_Carpani', ascending=True)
axes[1].barh(synergy_df['Hastalik'], synergy_df['Risk_Carpani'], color='#d95f02', edgecolor='black', alpha=0.85)
axes[1].axvline(1.0, color='black', linestyle='--', label='Yansiz Cizgi (1.0x)')
axes[1].set_title('ACL Yirtigi Varlıgında Diger Hastaliklarin Risk Carpani', weight='bold')
axes[1].set_xlabel('Risk Carpani (Kat)')
axes[1].legend()

plt.tight_layout()
plt.show()

display(synergy_df.sort_values(by='Risk_Carpani', ascending=False).reset_index(drop=True))

### Bulgular ve Modelleme Karari
1. **Biyomekanik Baglanti (Contusion):** ACL koptugunda kemik ezilmesi (Contusion) gorulme riski **3.07 katina (%54.2)** cikmaktadir. Bu durum literatürdeki *Pivot-Shift Bone Bruise* oruntusunu dogrular.
2. **Ikincil Hasar (Fracture & Effusion):** ACL yirtiklarinda kirik riski 2.23 katina, efuzyon riski ise %75.0'a yukselmektedir.
3. **Ters Korelasyon (Osteoartrit / Medial OA):** ACL yirtiginda kireclenme riski **0.35 katina** dusmektedir. Nedeni, ACL travmalarinin genc sporcularda akut olmasi, OA'nin ise ileri yas dejenerasyonu olmasidir.
- **Karar:** 12 ayri bagimsiz ikili siniflandirma yerine, etiketler arasi bu capraz bagimliligi ogrenebilen **Cok Gorevli Ortak Ag (Shared Representation Multi-Label Architecture)** kullanilmalidir.

## Bolum 3: Cok Dilli Klinik Metin Adli Bilimi

### Metod ve Arac
4.349 etiketsiz vakanin serbest metin `Report` alanlari leksikal oruntu tarayicilari ve diller arasi tip terimleri sozlukleri ile siniflandirildi. Metin uzunlugu, belirtec (token) sayilari ve tipik medikal kaliplar incelendi.

### Klinik ve Matematiksel Mantik
Raporlarin tek bir dilde (ornegin yalnizca Ispanyolca) oldugu varsayimi ile regex veya sozluk kural motoru yazilirsa, farkli dillerdeki olumsuzluk kaliplari (*"no"*, *"sin"*, *"aucun"*, *"zonder"*, *"intact"*) kural tabanini tamamen bozar. Bu durum bilgi kaybina ve yalanci etiket (label noise) patlamasina yol acar.

In [ ]:
def classify_language(text):
    t = str(text).lower()
    if any(k in t for k in [' the ', 'normal', 'tear', 'intact', 'compartment', 'effusion']):
        return 'English'
    elif any(k in t for k in [' de ', 'rotura', 'derrame', 'rodilla', 'menisco', 'edema']):
        return 'Spanish'
    elif any(k in t for k in ['aucun', 'fracture', 'dégénératif', 'articulaire', 'droite']):
        return 'French'
    elif any(k in t for k in ['knie', 'rechts', 'inlichtingen', 'zonder', 'meniscusscheur']):
        return 'Dutch / German'
    return 'Diger / Karma'

train_df['detected_lang'] = train_df['Report'].apply(classify_language)
lang_dist = train_df['detected_lang'].value_counts()

plt.figure(figsize=(10, 4.5))
bars = plt.bar(lang_dist.index, lang_dist.values, color=['#3182bd', '#6baed6', '#9ecae1', '#c6dbef', '#e6550d'], edgecolor='black')
plt.title('Radyoloji Raporlarinin Dil Kokeni Dagilimi', weight='bold')
plt.ylabel('Rapor Sayisi')
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 30, f"{yval}\n(%{yval/len(train_df)*100:.1f})", ha='center', va='bottom', weight='bold')
plt.ylim(0, max(lang_dist.values) + 300)
plt.tight_layout()
plt.show()

### Farkli Dillerden Ornek Klinik Rapor Kesitleri

In [ ]:
for lang in ['English', 'Spanish', 'French', 'Dutch / German']:
    sample = train_df[train_df['detected_lang'] == lang]['Report'].iloc[0]
    clean_sample = ' '.join(sample.split()[:45]) + '...'
    print(f"[{lang.upper()} RAPOR ORNEGI]:\n{clean_sample}\n")

### Bulgular ve Modelleme Karari
- **Cok Dillilik Gercegi:** Raporlar tek dilli degildir: Ingilizce (%43), Fransizca (%24), Ispanyolca (%17) ve Flemenkce/Almanca (%16).
- **Regex'in Imkansizligi:** Her dilde tip terimleri, olumsuzluk ekleri ve kisaltmalar farklidir. Kural tabanli bir yontem basarisizliga mahkumdur.
- **Karar:** Raporlardan 12 etiketi cikarmak icin kesinlikle **Cok Dilli Dil Modelleri (Multilingual E5, Qwen, Llama-3)** veya acik kaynak **LLM Distillation** etiketleri kullanilacaktir.

## Bolum 4: Manyetik Rezonans Fizigi — Sivi ve Yag Baskilama Sekanslari

### Metod ve Arac
`train_series.csv` uzerinde `Anatomical_Plane`, `Fluid_Sensitive` (T2/PD agirlikli) ve `Fat_Suppression` (FS/STIR) alanlari arasinda capraz frekans analizi calistirildi.

### Klinik ve Matematiksel Mantik
MR goruntulemede doku kontrastini RF puls sekanslari belirler:
- **Fluid-Sensitive (Siviya Duyarli):** T2 ve Proton Yoğunluğu (PD) sekanslarinda serbest su molekulleri bembeyaz (hiperintens) gorunur. Yirtik alani, eklem sivisi ve iltihap bu serilerde teshis edilir.
- **Fat-Suppression (Yag Baskilama):** Kemik iligindeki sari yag sinyali sondurulur, boylece kemigin icindeki odem ve ezilme (contusion) aciga cikar.

In [ ]:
fs_ct = pd.crosstab(
    train_series['Anatomical_Plane'],
    [train_series['Fluid_Sensitive'], train_series['Fat_Suppression']],
    rownames=['Anatomik Duzlem'],
    colnames=['Fluid_Sensitive', 'Fat_Suppression']
)

print("Duzlem Bazinda Sivi ve Yag Baskilama Frekans Tablosu:")
display(fs_ct)

# Hasta basina ortalama sivi duyarli seri sayisi
fs_study_avg = train_series.groupby('StudyInstanceUID')['Fluid_Sensitive'].sum().mean()
print(f"\nHasta basina ortalama Fluid-Sensitive & Fat-Suppressed seri sayisi: {fs_study_avg:.2f}")

### Bulgular ve Modelleme Karari
- Her hastada ortalama **3.2 adet sivi duyarlı ve yag baskilamali seri** mevcuttur.
- **Karar:** Ozellikle `Effusion`, `Synovitis` ve `Contusion` tahminlerinde modelin goruntu ozelliklerine bu sekans sinyalleri ek meta-veri kanali olarak entegre edilmelidir.

## Bolum 5: 2.5D Pencereleme (Windowing) Geometrisi ve Dilim Havuzlama

### Metod ve Arac
3 boyutlu bir MR serisinde ardisik uc kesitin $[S_{i-1}, S_i, S_{i+1}]$ seklinde RGB renk kanallarina dizilmesi ve $Z$-ekseni koordinatina gore fiziksel siralanmasi algoritmasi.

### Klinik ve Matematiksel Mantik
- **Neden Saf 3D Degil?** 3D konvolusyonlar parametre patlamasina ve asiri bellek tuketimine yol acar; ImageNet ile on-egitilmis agirliklardan faydalanamaz.
- **Neden Saf 2D Degil?** 2D modeller lezyonun derinlik boyunca ilerleyisini (slice-to-slice continuity) yakalayamaz.
- **Neden Average Pooling Yasak?** Bir meniskus yirtigi 30 dilimlik bir seride yalnizca **2 veya 3 dilimde** gorunur. Dilim ozelliklerinin ortalamasi alinirsa yirtik sinyali saglikli dilimler icinde seyreltilir (dilution effect). Bu nedenle **Finding-Specific Attention Pooling** zorunludur.

In [ ]:
def simulate_25d_stack(num_slices=32, target_windows=20):
    """
    32 dilimlik bir MR hacminde uc bolgelerdeki gurultu dilimlerini temizleyip
    ardisik 3'lu RGB pencereleri olusturan ornekleme fonksiyonu.
    """
    # Uc gurultuleri temizleme (%10 alt, %10 ust sinir)
    start = int(num_slices * 0.10)
    end = int(num_slices * 0.90)
    core_slices = list(range(start, end))
    
    # 3'lu pencereler
    windows = []
    for i in range(len(core_slices) - 2):
        windows.append((core_slices[i], core_slices[i+1], core_slices[i+2]))
        
    return core_slices, windows

core, windows = simulate_25d_stack(num_slices=32)
print(f"Ham Dilim Sayisi              : 32")
print(f"Cekirdek Anatomik Dilim Sayisi: {len(core)} (Indeksler: {core[0]} - {core[-1]})")
print(f"Uretilen 2.5D RGB Pencere     : {len(windows)}")
print(f"Ilk 3 RGB Dilim Grubu         : {windows[:3]}")

## Bolum 6: Modelleme Blueprint ve Stratejik Kararlar

Tum analitik ve klinik bulgularimizin nihai modelleme mimarimize yansimasi asagidaki tabloda ozetlenmistir:

| Analiz Alani | Kesfedilen Somut Gercek | Neden Kritik? | Uygulanacak Modelleme Karari |
| :--- | :--- | :--- | :--- |
| **Duzlem Tamligi** | Hastalarin %100'unde Sagittal + Coronal + Axial vardir. | Eksik veri yonetimine gerek kalmaz. | 3 duzlemi birlestiren **Multi-View Fusion** mimarisi. |
| **Klinik Sinerji** | ACL yirtigi Kemik Ezilmesi riskini 3.07 katina cikarirken Kireclenmeyi dusurur. | Bagimsiz modeller biyomekanik iliskileri kacirir. | Ortak temsilli **Cok Gorevli (Multi-Task) Omurga**. |
| **Rapor Dilleri** | Raporlar Ingilizce (%43), Fransizca (%24), Ispanyolca (%17), Flemenkce (%16). | Kural/regex kesinlikle coker. | **Cok Dilli LLM / Multilingual E5 Distillation** etiketleri. |
| **MR Sekans Fizigi** | Hasta basina 3.2 adet yag baskili sivi serisi duser. | Efuzyon ve ezilme bu serilerde parlar. | Sekans ozelliklerinin meta-veri olarak attention katmanina beslenmesi. |
| **Goruntu Temsili** | Lezyonlar sadece 2-3 kesitte lokalizedir. | Ortalama havuzlama lezyon sinyalini bogar. | **2.5D RGB Windowing + 12-Head Attention Pooling**. |

---